# Compute Joint UMAP Coordinates

Projects the **papers** and **teams** embeddings together with a single UMAP fit
and persists the 2D coordinates to `assets/reports/`. Every other reporting
notebook loads these coordinates, so the projection is computed **once** and the
geometry is identical across all figures.

> Run this **first**. Re-run only when the embeddings or topic models change.

In [1]:
# ── Make the shared aux/ package importable ───────────────────────────────────
# These notebooks live in 05-reporting/ alongside the aux/ package.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

In [2]:
import numpy as np
import pandas as pd

from aux.paths import EMBEDDINGS_DIR, MODELS_DIR, REPORTS_DIR, set_seed
from aux.coords import compute_joint_umap, save_coords

set_seed()

papers_emb = np.load(EMBEDDINGS_DIR / "papers_embeddings.npy")
teams_emb = np.load(EMBEDDINGS_DIR / "teams_embeddings.npy")
papers_dt = pd.read_csv(MODELS_DIR / "papers_doc_topics.txt", sep="\t")
teams_dt = pd.read_csv(MODELS_DIR / "teams_doc_topics.txt", sep="\t")

assert len(papers_emb) == len(papers_dt), "papers embeddings vs doc_topics mismatch"
assert len(teams_emb) == len(teams_dt), "teams embeddings vs doc_topics mismatch"
print(f"Papers: {len(papers_emb):,} | Teams: {len(teams_emb):,}")

Papers: 24,202 | Teams: 4,707


## Project both corpora together (single UMAP fit)

In [3]:
papers_xy, teams_xy = compute_joint_umap(papers_emb, teams_emb)
print(f"papers_xy {papers_xy.shape} | teams_xy {teams_xy.shape}")

/opt/anaconda3/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


papers_xy (24202, 2) | teams_xy (4707, 2)


## Persist coordinates

In [4]:
save_coords(
    papers_dt["id"].to_numpy(), papers_xy,
    teams_dt["UT"].to_numpy(), teams_xy,
)
print(f"Saved joint coordinates → {REPORTS_DIR}")
for f in ("joint_umap_papers_xy.tsv", "joint_umap_teams_xy.tsv"):
    print(f"  {f}")

Saved joint coordinates → /Users/cristian/Desktop/GitHub/igem-synbio/assets/reports
  joint_umap_papers_xy.tsv
  joint_umap_teams_xy.tsv
